# Part C — Battery Storage Impact (Denmark)

Runs two optimisations (with and without storage) and compares results.

1. **Section 1 (Optimisation)** — run once, saves results to `plots_part_c/`
2. **Section 2 (Plotting)** — loads from CSV, re-run freely without the solver

---
## Section 1 — Optimisation
*Run these cells once. Skip after CSVs exist.*

In [ ]:
import os
import pandas as pd
import pypsa

df_elec = pd.read_csv('data/electricity_demand.csv', sep=';', index_col=0)
df_elec.index = pd.to_datetime(df_elec.index)

df_onshorewind = pd.read_csv('data/onshore_wind_1979-2017.csv', sep=';', index_col=0)
df_onshorewind.index = pd.to_datetime(df_onshorewind.index)

df_offshorewind = pd.read_csv('data/offshore_wind_1979-2017.csv', sep=';', index_col=0)
df_offshorewind.index = pd.to_datetime(df_offshorewind.index)

df_solar = pd.read_csv('data/pv_optimal.csv', sep=';', index_col=0)
df_solar.index = pd.to_datetime(df_solar.index)

country = 'DNK'

def annuity(n, r):
    return r / (1. - 1. / (1. + r) ** n) if r > 0 else 1 / n

def build_network():
    net = pypsa.Network()
    hours_in_2015 = pd.date_range('2015-01-01 00:00Z', '2015-12-31 23:00Z', freq='h')
    net.set_snapshots(hours_in_2015.values)
    net.add("Bus", "electricity bus")
    net.add("Load", "load", bus="electricity bus", p_set=df_elec[country].values)
    net.add("Carrier", "gas", co2_emissions=0.19)
    net.add("Carrier", "onshorewind")
    net.add("Carrier", "offshorewind")
    net.add("Carrier", "solar")
    net.add("Carrier", "battery storage")

    CF_wind = df_onshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in net.snapshots]]
    net.add("Generator", "Onshore wind", bus="electricity bus",
            p_nom_extendable=True, carrier="onshorewind",
            capital_cost=annuity(27, 0.07) * 1118775, marginal_cost=0,
            p_max_pu=CF_wind.values)

    CF_wind_off = df_offshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in net.snapshots]]
    net.add("Generator", "Offshore wind", bus="electricity bus",
            p_nom_extendable=True, carrier="offshorewind",
            capital_cost=annuity(27, 0.07) * 2115944, marginal_cost=0,
            p_max_pu=CF_wind_off.values)

    CF_solar = df_solar[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in net.snapshots]]
    net.add("Generator", "Solar", bus="electricity bus",
            p_nom_extendable=True, carrier="solar",
            capital_cost=annuity(25, 0.07) * 450000, marginal_cost=0,
            p_max_pu=CF_solar.values)

    net.add("Generator", "OCGT", bus="electricity bus",
            p_nom_extendable=True, carrier="gas",
            capital_cost=annuity(25, 0.07) * 453960, marginal_cost=30 / 0.41)

    net.add("Generator", "CCGT", bus="electricity bus",
            p_nom_extendable=True, carrier="gas",
            capital_cost=annuity(25, 0.07) * 880000, marginal_cost=30 / 0.56)
    return net

In [ ]:
net_a = build_network()
net_a.optimize(solver_name='gurobi', solver_options={"OutputFlag": 0, "LogToConsole": 0})
print(f"Without storage — cost: {net_a.objective / 1e6:.2f} M\u20ac/yr")
print(net_a.generators.p_nom_opt.div(1e3))

In [ ]:
net_b = build_network()
net_b.add("StorageUnit", "battery storage",
          bus="electricity bus", carrier="battery storage",
          max_hours=2, capital_cost=annuity(20, 0.07) * 2 * 288000,
          efficiency_store=0.98, efficiency_dispatch=0.97,
          p_nom_extendable=True, cyclic_state_of_charge=True)
net_b.optimize(solver_name='gurobi', solver_options={"OutputFlag": 0, "LogToConsole": 0})
print(f"With storage — cost: {net_b.objective / 1e6:.2f} M\u20ac/yr")
print(f"Cost reduction: {(net_a.objective - net_b.objective) / 1e6:.2f} M\u20ac/yr")
print(f"Battery capacity: {net_b.storage_units.p_nom_opt['battery storage'] / 1e3:.2f} GW")

In [ ]:
os.makedirs('plots_part_c', exist_ok=True)

# Without storage
net_a.generators.p_nom_opt.to_frame('p_nom_opt').to_csv('plots_part_c/no_storage_capacity.csv')
net_a.generators_t.p.to_csv('plots_part_c/no_storage_dispatch.csv')

# With storage
cap_with = net_b.generators.p_nom_opt.copy()
cap_with['battery storage'] = net_b.storage_units.p_nom_opt['battery storage']
cap_with.to_frame('p_nom_opt').to_csv('plots_part_c/with_storage_capacity.csv')
net_b.generators_t.p.to_csv('plots_part_c/with_storage_dispatch.csv')
net_b.storage_units_t.p.to_csv('plots_part_c/battery_p.csv')
net_b.storage_units_t.state_of_charge.to_csv('plots_part_c/battery_soc.csv')
net_b.loads_t.p.to_csv('plots_part_c/load.csv')

# Summary comparison
pd.Series({'no_storage': net_a.objective / 1e6, 'with_storage': net_b.objective / 1e6},
          name='cost_M_eur').to_csv('plots_part_c/system_costs.csv')
print("Saved CSVs to plots_part_c/")

---
## Section 2 — Plotting
*Start here after CSVs exist. No solver needed.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cap_no  = pd.read_csv('plots_part_c/no_storage_capacity.csv',    index_col=0)['p_nom_opt']
cap_yes = pd.read_csv('plots_part_c/with_storage_capacity.csv',  index_col=0)['p_nom_opt']
dispatch_no  = pd.read_csv('plots_part_c/no_storage_dispatch.csv',   index_col=0, parse_dates=True)
dispatch_yes = pd.read_csv('plots_part_c/with_storage_dispatch.csv', index_col=0, parse_dates=True)
battery_p    = pd.read_csv('plots_part_c/battery_p.csv',             index_col=0, parse_dates=True)['battery storage']
battery_soc  = pd.read_csv('plots_part_c/battery_soc.csv',           index_col=0, parse_dates=True)['battery storage']
load         = pd.read_csv('plots_part_c/load.csv',                  index_col=0, parse_dates=True)['load']
costs        = pd.read_csv('plots_part_c/system_costs.csv',          index_col=0)['cost_M_eur']

GEN_NAMES  = ['Onshore wind', 'Offshore wind', 'Solar', 'OCGT', 'CCGT']
GEN_LABELS = ['Onshore wind', 'Offshore wind', 'Solar PV', 'Gas (OCGT)', 'Gas (CCGT)']
GEN_COLORS = ['blue', 'dodgerblue', 'orange', 'crimson', 'darkviolet']

print(f"System cost without storage: {costs['no_storage']:.1f} M\u20ac/yr")
print(f"System cost with storage:    {costs['with_storage']:.1f} M\u20ac/yr")
print(f"Saving from storage:         {costs['no_storage'] - costs['with_storage']:.1f} M\u20ac/yr")
print(f"Battery installed:           {cap_yes.get('battery storage', 0) / 1e3:.2f} GW")

In [ ]:
ALL_NAMES  = GEN_NAMES + ['battery storage']
ALL_LABELS = GEN_LABELS + ['Battery storage']
ALL_COLORS = GEN_COLORS + ['lightgreen']

cap_sizes = [cap_yes.get(n, 0) for n in ALL_NAMES]
plt.figure(figsize=(6, 5), dpi=150)
plt.pie(cap_sizes, colors=ALL_COLORS,
        labels=[f'{l}\n{s/1e3:.1f} GW' for l, s in zip(ALL_LABELS, cap_sizes)],
        wedgeprops={'linewidth': 0})
plt.axis('equal')
plt.title('Installed capacity mix (with battery storage)', y=1.05, fontweight='bold')
plt.tight_layout()
plt.savefig('plots_part_c/1c_installed_capacity_mix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
bat_dis = battery_p.clip(lower=0)
bat_ch  = battery_p.clip(upper=0)
renewables = dispatch_yes[['Onshore wind', 'Offshore wind', 'Solar']].sum(axis=1)

supply = pd.concat([
    bat_dis.rename("Battery discharge"),
    renewables.rename("Renewables"),
], axis=1)

fig, ax = plt.subplots(figsize=(12, 4), dpi=150)
supply.plot.area(ax=ax, linewidth=0, stacked=True, color=["#2ca02c", "#1f77b4"])
bat_ch.rename("Battery charge").to_frame().plot.area(
    ax=ax, linewidth=0, stacked=True, color=["#ff7f0e"])
load.plot(ax=ax, color="black", linestyle="--", label="Demand")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("MW")
ax.set_title("Battery storage behaviour — full year")
ax.legend(frameon=False, bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig('plots_part_c/1c_battery_stackplot_year.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), dpi=150)
for ax, (start, end, season) in zip(axes, [(0, 168, 'Winter (Jan)'), (4344, 4512, 'Summer (Jul)')]):
    battery_soc.iloc[start:end].plot(ax=ax, color='lightgreen', lw=2)
    ax.set_ylabel('State of charge [MWh]')
    ax.set_title(f'Battery state of charge — {season}')
    ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('plots_part_c/1c_battery_soc.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for start, end, season, fname in [
    (0,    168,  'Winter (Jan)', '1c_dispatch_winter.png'),
    (4344, 4512, 'Summer (Jul)', '1c_dispatch_summer.png'),
]:
    active = [n for n in GEN_NAMES if cap_yes.get(n, 0) > 0]
    active_labels = [l for n, l in zip(GEN_NAMES, GEN_LABELS) if cap_yes.get(n, 0) > 0]
    active_colors = [c for n, c in zip(GEN_NAMES, GEN_COLORS) if cap_yes.get(n, 0) > 0]

    fig, ax = plt.subplots(figsize=(10, 4), dpi=150)
    stack_data = [dispatch_yes[n].iloc[start:end].values for n in active]
    ax.stackplot(range(end - start), *stack_data,
                 labels=active_labels, colors=active_colors, linewidth=0)

    bat_dis_w = battery_p.clip(lower=0).iloc[start:end]
    bat_ch_w  = battery_p.clip(upper=0).iloc[start:end]
    ax.fill_between(range(end - start), bat_dis_w.values, label='Battery discharge', color='#2ca02c', alpha=0.8)
    ax.fill_between(range(end - start), bat_ch_w.values,  label='Battery charge',    color='#ff7f0e', alpha=0.8)
    ax.plot(load.iloc[start:end].values, color='black', lw=2, label='Demand')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel('MW')
    ax.set_xlabel('Hour of week')
    ax.set_title(f'Dispatch with battery storage — {season}')
    ax.legend(frameon=False, bbox_to_anchor=(1.05, 1), fontsize=8)
    plt.tight_layout()
    plt.savefig(f'plots_part_c/{fname}', dpi=300, bbox_inches='tight')
    plt.show()